In [4]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score,confusion_matrix)
from feature_engineering import build_pipeline


In [13]:
df=pd.read_csv("https://raw.githubusercontent.com/awais-DS/Data/refs/heads/main/WA_Fn-UseC_-Telco-Customer-Churn.csv")

In [14]:
df=df[["gender","tenure","Contract","Dependents","MonthlyCharges","Churn"]]

In [15]:
x=df.drop(columns=["Churn"])
y=df["Churn"].map({"No":0,"Yes":1})

In [16]:
x.head()

,gender,tenure,Contract,Dependents,MonthlyCharges
0,Female,1,Month-to-month,No,29.85
1,Male,34,One year,No,56.95
2,Male,2,Month-to-month,No,53.85
3,Male,45,One year,No,42.30
4,Female,2,Month-to-month,No,70.70


In [17]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,stratify=y,random_state=0)

In [18]:
print(x_train.shape,y_train.shape)

(5634, 5) (5634,)


In [22]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.linear_model import LogisticRegression

def build_pipeline():
    encoders = ColumnTransformer(
        transformers=[
            ("gender", OneHotEncoder(drop="first", handle_unknown="ignore"), ["gender"]),
            ("dependents", OneHotEncoder(drop="first", handle_unknown="ignore"), ["Dependents"]),
            ("contract", OrdinalEncoder(
                categories=[["Month-to-month", "One year", "Two year"]]), ["Contract"]),
        ],
        remainder="passthrough",
    )
    return Pipeline([
        ("encoders", encoders),
        ("model", LogisticRegression(class_weight="balanced",
                                     max_iter=1000, random_state=0)),
    ])

In [23]:
pipeline=build_pipeline()

In [24]:
pipeline.fit(x_train,y_train)

,steps,"[('encoders', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('gender', ...), ('dependents', ...), ...]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [25]:
pred = pipeline.predict(x_test)
print(f"Accuracy : {accuracy_score(y_test, pred):.4f}")
print(f"Precision: {precision_score(y_test, pred):.4f}")
print(f"Recall   : {recall_score(y_test, pred):.4f}")
print(f"F1 Score : {f1_score(y_test, pred):.4f}")
print(confusion_matrix(y_test, pred))

Accuracy : 0.7331
Precision: 0.4983
Recall   : 0.8075
F1 Score : 0.6163
[[731 304]
 [ 72 302]]


In [ ]:
print(len(df))                  # rows before
df = df.drop_duplicates()   # or whatever cleaning i did in the notebook
print(len(df)) 

7043
6884


In [27]:
import os 
os.makedirs("models",exist_ok=True)
joblib.dump(pipeline,"models/model.pkl")
print("saved")

saved
